# Virtual Fitting Room API for Google Colab

Run all cells. The final cell prints a public `/tryon` URL that you can paste into the ASP.NET MVC app as `VirtualTryOn__ApiUrl`.

This notebook is a free lightweight overlay backend. It does not use Replicate credit.

In [ ]:
!pip -q install fastapi uvicorn python-multipart pillow opencv-python-headless numpy
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

## Public tunnel

This notebook uses Cloudflare Tunnel to create a temporary public URL. No ngrok token is required.

In [ ]:
# No token is needed for Cloudflare Tunnel.

In [ ]:
%%writefile colab_tryon_api.py
import base64
import io

import cv2
import numpy as np
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image

app = FastAPI(title="Virtual Fitting Room Colab API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


def fit_image(image: Image.Image, max_side: int) -> Image.Image:
    width, height = image.size
    largest_side = max(width, height)
    if largest_side <= max_side:
        return image
    scale = max_side / largest_side
    new_size = (max(1, int(width * scale)), max(1, int(height * scale)))
    return image.resize(new_size, Image.Resampling.LANCZOS)


def pil_to_bgr(image: Image.Image) -> np.ndarray:
    return cv2.cvtColor(np.array(image.convert("RGB")), cv2.COLOR_RGB2BGR)


def bgr_to_pil(image: np.ndarray) -> Image.Image:
    return Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))


def remove_white_background(garment_bgr: np.ndarray):
    garment_rgb = cv2.cvtColor(garment_bgr, cv2.COLOR_BGR2RGB)
    lower = np.array([0, 0, 0], dtype=np.uint8)
    upper = np.array([242, 242, 242], dtype=np.uint8)
    non_white_mask = cv2.inRange(garment_rgb, lower, upper)

    hsv = cv2.cvtColor(garment_bgr, cv2.COLOR_BGR2HSV)
    low_sat_mask = cv2.inRange(hsv, np.array([0, 0, 200]), np.array([180, 40, 255]))
    mask = cv2.bitwise_and(non_white_mask, cv2.bitwise_not(low_sat_mask))

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.GaussianBlur(mask, (5, 5), 0)

    coords = cv2.findNonZero(mask)
    if coords is None:
        full_mask = np.full(garment_bgr.shape[:2], 255, dtype=np.uint8)
        return garment_bgr, full_mask

    x, y, w, h = cv2.boundingRect(coords)
    return garment_bgr[y:y + h, x:x + w], mask[y:y + h, x:x + w]


def detect_face(person_bgr: np.ndarray):
    gray = cv2.cvtColor(person_bgr, cv2.COLOR_BGR2GRAY)
    cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(cascade_path)
    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    if len(faces) == 0:
        return None
    return max(faces, key=lambda face: face[2] * face[3])


def estimate_upper_body_quad(person_bgr: np.ndarray) -> np.ndarray:
    height, width = person_bgr.shape[:2]
    face = detect_face(person_bgr)
    if face is not None:
        x, y, w, h = face
        center_x = x + (w / 2.0)
        shoulders_y = y + (h * 1.95)
        waist_y = y + (h * 5.25)
        top_half_width = w * 1.2
        bottom_half_width = top_half_width * 0.78
    else:
        center_x = width * 0.5
        shoulders_y = height * 0.21
        waist_y = height * 0.66
        top_half_width = width * 0.17
        bottom_half_width = width * 0.14

    shoulders_y = max(0.0, min(float(height - 1), shoulders_y))
    waist_y = max(shoulders_y + 30.0, min(float(height - 1), waist_y))
    quad = np.array([
        [center_x - top_half_width, shoulders_y],
        [center_x + top_half_width, shoulders_y],
        [center_x + bottom_half_width, waist_y],
        [center_x - bottom_half_width, waist_y],
    ], dtype=np.float32)
    quad[:, 0] = np.clip(quad[:, 0], 0, width - 1)
    quad[:, 1] = np.clip(quad[:, 1], 0, height - 1)
    return quad


def estimate_lower_body_quad(person_bgr: np.ndarray) -> np.ndarray:
    height, width = person_bgr.shape[:2]
    center_x = width * 0.5
    quad = np.array([
        [center_x - width * 0.20, height * 0.52],
        [center_x + width * 0.20, height * 0.52],
        [center_x + width * 0.16, height * 0.96],
        [center_x - width * 0.16, height * 0.96],
    ], dtype=np.float32)
    quad[:, 0] = np.clip(quad[:, 0], 0, width - 1)
    quad[:, 1] = np.clip(quad[:, 1], 0, height - 1)
    return quad


def estimate_target_quad(person_bgr: np.ndarray, category: str) -> np.ndarray:
    if category == "lower":
        return estimate_lower_body_quad(person_bgr)
    if category == "overall":
        upper = estimate_upper_body_quad(person_bgr)
        lower = estimate_lower_body_quad(person_bgr)
        return np.array([upper[0], upper[1], lower[2], lower[3]], dtype=np.float32)
    return estimate_upper_body_quad(person_bgr)


def warp_garment_to_person(person_bgr: np.ndarray, garment_bgr: np.ndarray, category: str) -> np.ndarray:
    garment_bgr, garment_mask = remove_white_background(garment_bgr)
    person_h, person_w = person_bgr.shape[:2]
    garment_h, garment_w = garment_bgr.shape[:2]

    if category == "lower":
        src_points = np.array([[garment_w * 0.24, garment_h * 0.06], [garment_w * 0.76, garment_h * 0.06], [garment_w * 0.66, garment_h * 0.98], [garment_w * 0.34, garment_h * 0.98]], dtype=np.float32)
    elif category == "overall":
        src_points = np.array([[garment_w * 0.18, garment_h * 0.12], [garment_w * 0.82, garment_h * 0.12], [garment_w * 0.70, garment_h * 0.98], [garment_w * 0.30, garment_h * 0.98]], dtype=np.float32)
    else:
        src_points = np.array([[garment_w * 0.16, garment_h * 0.18], [garment_w * 0.84, garment_h * 0.18], [garment_w * 0.72, garment_h * 0.98], [garment_w * 0.28, garment_h * 0.98]], dtype=np.float32)

    dst_points = estimate_target_quad(person_bgr, category)
    matrix = cv2.getPerspectiveTransform(src_points, dst_points)
    warped_garment = cv2.warpPerspective(garment_bgr, matrix, (person_w, person_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    warped_mask = cv2.warpPerspective(garment_mask, matrix, (person_w, person_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    solid_mask = cv2.threshold(warped_mask, 24, 255, cv2.THRESH_BINARY)[1]
    shadow_mask = cv2.dilate(solid_mask, np.ones((11, 11), np.uint8), iterations=1)
    shadow_mask = cv2.GaussianBlur(shadow_mask, (21, 21), 0).astype(np.float32) / 255.0
    shadow = person_bgr.astype(np.float32) * (1.0 - (shadow_mask[..., None] * 0.16))
    shadow = np.clip(shadow, 0, 255).astype(np.uint8)
    alpha = cv2.GaussianBlur(solid_mask, (5, 5), 0).astype(np.float32)[..., None] / 255.0
    alpha = np.clip(alpha, 0.0, 1.0)
    composite = warped_garment.astype(np.float32) * alpha + shadow.astype(np.float32) * (1.0 - alpha)
    return np.clip(composite, 0, 255).astype(np.uint8)


def normalize_category(category: str) -> str:
    value = (category or "upper").strip().lower()
    if value in {"pants", "lower", "lower_body"}:
        return "lower"
    if value in {"dress", "dresses", "overall"}:
        return "overall"
    return "upper"


def run_tryon(person_bytes: bytes, cloth_bytes: bytes, category: str) -> bytes:
    person_image = fit_image(Image.open(io.BytesIO(person_bytes)).convert("RGB"), 900)
    cloth_image = fit_image(Image.open(io.BytesIO(cloth_bytes)).convert("RGB"), 700)
    person_bgr = pil_to_bgr(person_image)
    cloth_bgr = pil_to_bgr(cloth_image)
    result_bgr = warp_garment_to_person(person_bgr, cloth_bgr, normalize_category(category))
    output = io.BytesIO()
    bgr_to_pil(result_bgr).save(output, format="PNG")
    return output.getvalue()


@app.get("/health")
def health():
    return {"status": "ok", "engine": "colab-fast-overlay"}


@app.post("/tryon")
async def tryon(
    person_image: UploadFile = File(...),
    garment_image: UploadFile = File(...),
    category: str = Form("upper"),
):
    output = run_tryon(await person_image.read(), await garment_image.read(), category)
    return {"outputImageBase64": base64.b64encode(output).decode("utf-8")}


In [ ]:
import re
import subprocess
import threading
import time

subprocess.Popen(["python", "-m", "uvicorn", "colab_tryon_api:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(4)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    print(line, end="")
    match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Cloudflare tunnel URL was not created. Re-run this cell.")

print("\nHealth URL:", public_url + "/health")
print("Paste this into VirtualTryOn__ApiUrl:")
print(public_url + "/tryon")

def keep_logs_open():
    for line in tunnel.stdout:
        print(line, end="")

threading.Thread(target=keep_logs_open, daemon=True).start()